# 02 - Kink Calibration

**Purpose.** Fetch the on-chain kink parameters for Aave v3 USDC and Compound v3 cUSDCv3, then verify that the deterministic `f_kink(u)` function explains most of the rate level so that the dual-branch forecaster (Branch A) only has to learn the bounded mean-zero residual.

**Prerequisites.**
- `data.fetch_kink_params` (requires `ETHEREUM_RPC_URL`); gracefully   falls back to plan-default values if no RPC is available.
- `data.features.f_kink`.

**Expected runtime.** < 1 minute.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


## 1. Fetch kink params (with graceful fallback)

In [ ]:
from data.features import AaveKinkParams, CompoundKinkParams, f_kink
import os

# Plan defaults (PROJECT_2_PLAN.md S5; pre-set if no RPC is available).
DEFAULT_AAVE = AaveKinkParams(
    base_variable_borrow_rate=0.0,
    slope1=0.05, slope2=0.60,
    optimal_usage_ratio=0.92, reserve_factor=0.10,
)
DEFAULT_COMP = CompoundKinkParams(
    supply_kink=0.85,
    supply_per_second_base=0.0,
    supply_per_second_slope_low=0.04,
    supply_per_second_slope_high=0.50,
)

kink_aave, kink_comp = DEFAULT_AAVE, DEFAULT_COMP
if os.environ.get('ETHEREUM_RPC_URL'):
    try:
        from data.fetch_kink_params import (
            fetch_aave_usdc_kink, fetch_compound_usdc_kink,
        )
        rpc = os.environ['ETHEREUM_RPC_URL']
        kink_aave = fetch_aave_usdc_kink(rpc)
        kink_comp = fetch_compound_usdc_kink(rpc)
        print('[real] fetched live kink params from chain')
    except Exception as e:
        print(f'[fallback] RPC fetch failed: {e}; using defaults')
else:
    print('[fallback] ETHEREUM_RPC_URL not set; using plan defaults')

print('Aave    :', kink_aave)
print('Compound:', kink_comp)


## 2. Plot f_kink(u) for both protocols

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

u_grid = np.linspace(0.0, 0.99, 200)
r_a = f_kink(u_grid, kink_aave)
r_c = f_kink(u_grid, kink_comp)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax[0].plot(u_grid, np.asarray(r_a) * 100, color='C0', label='f_kink Aave')
ax[0].axvline(kink_aave.optimal_usage_ratio, ls='--', color='gray',
              label=f'U_opt={kink_aave.optimal_usage_ratio:.2f}')
ax[0].set_title('Aave v3 USDC supply curve')
ax[0].set_xlabel('utilization u')
ax[0].set_ylabel('annualised supply APY %')
ax[0].legend()

ax[1].plot(u_grid, np.asarray(r_c) * 100, color='C1', label='f_kink Compound')
ax[1].axvline(kink_comp.supply_kink, ls='--', color='gray',
              label=f'kink={kink_comp.supply_kink:.2f}')
ax[1].set_title('Compound v3 USDC supply curve')
ax[1].set_xlabel('utilization u')
ax[1].legend()
plt.tight_layout()
plt.show()

# Caption: Protocol supply-rate functions. Note the slope discontinuity
# at the kink. This is exactly the structure exploited by the dual-branch
# Branch-A residual target (PROJECT_2_PLAN.md S2.2).


## 3. Overlay (utilization, rate) scatter on top of f_kink

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax[0].scatter(df['u_aave'], df['r_aave'] * 100, s=4, alpha=0.2, color='C0')
ax[0].plot(u_grid, np.asarray(r_a) * 100, color='black', label='f_kink')
ax[0].set_title('Aave: empirical vs kink')
ax[0].set_xlabel('u_aave'); ax[0].set_ylabel('APY %')
ax[0].legend()

ax[1].scatter(df['u_compound'], df['r_compound'] * 100, s=4, alpha=0.2, color='C1')
ax[1].plot(u_grid, np.asarray(r_c) * 100, color='black', label='f_kink')
ax[1].set_title('Compound: empirical vs kink')
ax[1].set_xlabel('u_compound')
ax[1].legend()
plt.tight_layout()
plt.show()

# Caption: Empirical (u, r) clouds on top of the theoretical curve.
# A good calibration leaves only a thin, mean-zero band around f_kink.


## 4. Verify residuals are mean-zero and bounded

In [ ]:
from data.features import rate_residual

eps_a = rate_residual(df['r_aave'], df['u_aave'], kink_aave)
eps_c = rate_residual(df['r_compound'], df['u_compound'], kink_comp)

print(f'Aave residual:    mean={eps_a.mean():+.5f}  std={eps_a.std():.5f}  '
      f"q01={np.quantile(eps_a, 0.01):+.5f}  q99={np.quantile(eps_a, 0.99):+.5f}")
print(f'Compound residual: mean={eps_c.mean():+.5f}  std={eps_c.std():.5f}  '
      f"q01={np.quantile(eps_c, 0.01):+.5f}  q99={np.quantile(eps_c, 0.99):+.5f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(eps_a, bins=60, color='C0', alpha=0.8, edgecolor='black')
ax[0].axvline(0, ls='--', color='red')
ax[0].set_title('Aave residual eps = r - f_kink(u)')
ax[1].hist(eps_c, bins=60, color='C1', alpha=0.8, edgecolor='black')
ax[1].axvline(0, ls='--', color='red')
ax[1].set_title('Compound residual eps = r - f_kink(u)')
plt.tight_layout()
plt.show()

# Caption: Residual histograms should be centred near zero. If means
# are large in magnitude, kink params are mis-calibrated for the panel.


## Next steps

- If means are not near zero, refresh `data/cached/kink_params.json`   via `python -m data.fetch_kink_params --force`.
- Proceed to `03_forecaster_training.ipynb` for the architecture smoke test.

Relevant plan section: **PROJECT_2_PLAN.md S2.2 (Forecast Component) and S5.1 (Calibration grid)**.
